# Belka Gen AI — Notebook 02: LoRA e agente

**Objetivo:** consolidar resultados **já executados e salvos** de treinamento LoRA, avaliação Qwen original × LoRA e comparação de cinco configurações da Belka. Este notebook **não treina modelos**, **não chama a API** e **não realiza novas inferências**.

**Fonte dos dados:** `outputs/belka-lora/training_run.json`, `avaliacao/resultados_lora.json`, `avaliacao/placar_comparacao_reais.json`, `avaliacao/resultados_comparacao_reais.json` e `treinamento/{train,val,test}.jsonl`.

Execute a partir da raiz do repositório ou da pasta `notebooks/`. As saídas abaixo foram geradas a partir dos artefatos do ZIP auditado; após atualizar os dados, execute as células novamente para conferir a consistência.

In [6]:
# Localiza o projeto e carrega os resultados já salvos.
import json
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "avaliacao" / "resultados_lora.json").exists():
    raise FileNotFoundError("Abra o notebook na pasta notebooks/ ou na raiz do repositório Belka")

def ler_json(caminho):
    return json.loads((ROOT / caminho).read_text(encoding="utf-8"))

registro_treino = ler_json("outputs/belka-lora/training_run.json")
lora = ler_json("avaliacao/resultados_lora.json")
placar = ler_json("avaliacao/placar_comparacao_reais.json")
comparacao = ler_json("avaliacao/resultados_comparacao_reais.json")
print("Arquivos lidos com sucesso.")
print("Registros de comparação:", len(comparacao))

Arquivos lidos com sucesso.
Registros de comparação: 85


## 1. Conjuntos de dados e configuração do treinamento

A contagem abaixo usa os arquivos JSONL que foram preservados no ZIP. A presença de exemplos semelhantes entre os conjuntos não é excluída por uma checagem de perguntas literalmente iguais.

In [7]:
# Conta os exemplos e mostra as configurações registradas do treino.
for nome in ("train", "val", "test"):
    linhas = (ROOT / "treinamento" / f"{nome}.jsonl").read_text(encoding="utf-8").splitlines()
    exemplos = [json.loads(linha) for linha in linhas if linha.strip()]
    print(f"{nome:>5}: {len(exemplos)} exemplos")
print("Modelo base:", registro_treino["base_model"])
for chave in ("device", "epochs", "rank", "lr", "grad_accum", "steps", "seed", "max_length"):
    print(f"{chave}: {registro_treino[chave]}")
print("Loss final de treinamento (registro):", round(registro_treino["loss_treino_final"], 4))

train: 138 exemplos
  val: 36 exemplos
 test: 36 exemplos
Modelo base: Qwen/Qwen2.5-1.5B-Instruct
device: cuda
epochs: 2
rank: 8
lr: 0.0001
grad_accum: 8
steps: 36
seed: 42
max_length: 768
Loss final de treinamento (registro): 0.9187


O registro `training_run.json` guarda a **loss final de treinamento**, mas não contém a série por época ou a loss de validação. Os valores de validação apresentados no relatório foram coletados na execução original, e não são recalculados por este notebook.

## 2. Qwen original × Qwen com LoRA — 36 casos de teste

A tabela reproduz métricas automáticas do `resultados_lora.json`. A comparação de tempo é baseada nos tempos registrados e **não** garante menor tempo de resposta em outros ambientes. A métrica `decisao_recusa_correta` exige JSON válido, conforme a avaliação corrigida.

In [8]:
# Compara as métricas do modelo original e do modelo com LoRA.
campos = [
    ("Total de casos", "total"),
    ("JSON válido", "json_valido"),
    ("Decisão de recusa correta (JSON válido)", "decisao_recusa_correta"),
    ("Recusas corretas (6 esperadas)", "recusas_corretas"),
    ("Citações exatas", "citacoes_exatas"),
    ("IDs de citação sem invenção", "citacoes_sem_inventar_ids"),
    ("Tempo médio (s)", "tempo_medio_segundos"),
]
a = lora["qwen_original"]["resumo"]
b = lora["qwen_lora"]["resumo"]
print(f"{'Métrica':<42} | {'Qwen original':>14} | {'Qwen + LoRA':>14}")
print("-" * 78)
for rotulo, campo in campos:
    x, y = a[campo], b[campo]
    if campo == "tempo_medio_segundos":
        x, y = f"{x:.4f}", f"{y:.4f}"
    print(f"{rotulo:<42} | {str(x):>14} | {str(y):>14}")

Métrica                                    |  Qwen original |    Qwen + LoRA
------------------------------------------------------------------------------
Total de casos                             |             36 |             36
JSON válido                                |              0 |             36
Decisão de recusa correta (JSON válido)    |              0 |             36
Recusas corretas (6 esperadas)             |              0 |              6
Citações exatas                            |              0 |             36
IDs de citação sem invenção                |              0 |             36
Tempo médio (s)                            |         3.9199 |         1.6843


**Interpretação:** o LoRA melhorou a respeito ao formato JSON e o registro exato de IDs no conjunto de teste. Essas métricas **não demonstram** correção do conteúdo das respostas. Na inspeção anterior foram observadas respostas excessivamente curtas e interpretações inadequadas, que permanecem como limitações do modelo.

## 3. Placar comparativo — cinco configurações, 17 casos por modo

O placar foi gerado a partir de execuções reais, com a consulta meteorológica em tempo real desativada para garantir uma comparação reproduzível.

O indicador `Recusas identificadas` verifica a presença de expressões que indicam uma recusa na resposta, mas não garante que ela esteja correta. Já o indicador `Gabarito citado` verifica se os IDs das citações correspondem aos esperados, sem garantir que as fontes realmente sustentem todo o conteúdo da resposta.

In [9]:
# Exibe o placar dos cinco modos e confere a quantidade de registros.
nomes = {
    "baseline": "Qwen puro",
    "rag": "Qwen + RAG",
    "rag_lora": "Qwen + RAG + LoRA",
    "agent": "Qwen + RAG + agente",
    "agent_lora": "Qwen + RAG + LoRA + agente",
}
print(f"{'Configuração':<29} | {'Execuções':>9} | {'JSON válido':>11} | {'Recusas':>8} | {'Gabarito':>8} | {'Falhas':>6}")
print("-" * 91)
for modo, nome in nomes.items():
    r = placar[modo]
    print(f"{nome:<29} | {r['executados']:>2}/{r['casos']:<6} | {r['json_valido_taxa']*100:>10.1f}% | {r['recusas_corretas']:>2}/5{'':<4} | {r['gabarito_citado']:>2}/12{'':<3} | {r['falhas_execucao']:>6}")
assert len(comparacao) == 85
assert all(sum(c["modo"] == modo for c in comparacao) == 17 for modo in nomes)
print("\nValidação: 85 registros, 17 por configuração.")

Configuração                  | Execuções | JSON válido |  Recusas | Gabarito | Falhas
-------------------------------------------------------------------------------------------
Qwen puro                     | 17/17     |       17.6% |  5/5     |  0/12    |      0
Qwen + RAG                    | 17/17     |       94.1% |  5/5     |  9/12    |      0
Qwen + RAG + LoRA             | 17/17     |      100.0% |  5/5     | 11/12    |      0
Qwen + RAG + agente           | 17/17     |      100.0% |  5/5     | 10/12    |      0
Qwen + RAG + LoRA + agente    | 17/17     |      100.0% |  4/5     | 10/12    |      0

Validação: 85 registros, 17 por configuração.


## 4. Amostras reais das respostas e limitações

Exibimos registros presentes no JSON comparativo para que a análise não fique restrita ao placar. O gabarito indicado por ID pode não ser a única evidência útil: por exemplo, em T06, o trecho K14 também sustenta algumas orientações, mesmo quando o gabarito aponta K06.

In [10]:
# Mostra respostas reais selecionadas para examinar suas limitações.
for caso_id, modo in (("T03", "baseline"), ("T12", "rag"), ("T17", "agent_lora"), ("T12", "agent_lora")):
    registro = next((x for x in comparacao if x["id"] == caso_id and x["modo"] == modo), None)
    if registro is None:
        print("Registro ausente:", caso_id, modo)
        continue
    print(f"\n{caso_id} / {modo}")
    print("Resposta:", registro.get("resposta"))
    print("Citações registradas:", registro.get("citacoes"))
    print("JSON válido:", registro.get("json_valido"), "| Gabarito citado:", registro.get("trecho_gabarito_encontrado"))


T03 / baseline
Resposta: Não consta na base consultada.
Citações registradas: 
JSON válido: False | Gabarito citado: False

T12 / rag
Resposta: A chuva prevista não garante enchentes. É importante considerar a vulnerabilidade local, o histórico de eventos e os alertas oficiais. A Defesa Civil deve ser consultada para orientações específicas.
Citações registradas: K02
JSON válido: True | Gabarito citado: False

T17 / agent_lora
Resposta: Observação dos sinais mencionados pode ajudar a identificar possíveis deslizamentos. Evite a área e procure ajuda da Defesa Civil se detectar qualquer mudança sutil no solo ou estrutura. Consulte a definição mais recente do CEMADEN sobre riscos de deslizamento para informações precisas.
Citações registradas: K03
JSON válido: True | Gabarito citado: False

T12 / agent_lora
Resposta: Uma previsão de chuva não garante enchentes. Avalie vulnerabilidades locais, acompanhamento da Defesa Civil e preparação adequada. Evite se aproximar de áreas suscetíveis a 

## 5. Conclusão e referências para auditoria

- Nas métricas automáticas disponíveis, **RAG + LoRA** alcançou 100% de JSON válido, 5/5 recusas identificadas e 11/12 gabaritos citados.
- A configuração com todos os componentes **não superou** RAG + LoRA nesses indicadores.
- A presença de uma citação **não demonstra** que a fonte realmente sustenta o conteúdo da resposta.
- O agente é controlado por regras, não autônomo; os indicadores do MVP não são alertas oficiais da Defesa Civil.
- Os cinco casos de erro e respectivas propostas de correção estão registrados em `docs/analise_cinco_erros.md`. São **propostas**, não melhorias testadas.

**Arquivos de reprodução:** `treinamento/treinar_lora.py`, `avaliacao/avaliar_lora.py`, `scripts/avaliar_geracao.py`, `avaliacao/resultados_lora.json`, `avaliacao/resultados_comparacao_reais.json` e `avaliacao/placar_comparacao_reais.json`.